In [1]:
import pandas as pd

In [ ]:
df = pd.read_csv('dataset_final.csv')

token = ''

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7071 entries, 0 to 7070
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   index               7071 non-null   int64 
 1   Title               7071 non-null   object
 2   Year                7071 non-null   int64 
 3   Event               7071 non-null   object
 4   Area                7071 non-null   object
 5   Abstract            7071 non-null   object
 6   Introduction        7071 non-null   object
 7   Conclusion          7071 non-null   object
 8   Class               7071 non-null   object
 9   Introduction_clean  7071 non-null   object
 10  Conclusion_clean    7071 non-null   object
dtypes: int64(2), object(9)
memory usage: 607.8+ KB


In [4]:
# ============================================================
# CONTAGEM DE TOKENS POR CLASSE
# - Polido_IA: usa Abstract
# - Gerado_IA: usa Title + Introduction + Conclusion
# - Não inclui tokens do prompt/comando
# ============================================================

import pandas as pd

# Instalar se necessário:
!pip install tiktoken

import tiktoken

   ---------------------------------------- 0.0/874.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/874.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/874.8 kB ? eta -:--:--
   ----------------------- ---------------- 524.3/874.8 kB 1.7 MB/s eta 0:00:01
   ---------------------------------------- 874.8/874.8 kB 2.2 MB/s  0:00:00



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:


# Escolha uma codificação compatível com modelos recentes
enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    if pd.isna(text):
        return 0
    return len(enc.encode(str(text)))

# Cópia de segurança
df_tokens = df.copy()

# ------------------------------------------------------------
# POLIDO_IA — conta tokens apenas do Abstract
# ------------------------------------------------------------

mask_polido = df_tokens["Class"].eq("Polida_IA")

df_tokens.loc[mask_polido, "input_text_for_tokens"] = (
    df_tokens.loc[mask_polido, "Abstract"].fillna("")
)

# ------------------------------------------------------------
# GERADO_IA — conta tokens de Title + Introduction + Conclusion
# ------------------------------------------------------------

mask_gerado = df_tokens["Class"].eq("Gerada")

df_tokens.loc[mask_gerado, "input_text_for_tokens"] = (
    "Title: " + df_tokens.loc[mask_gerado, "Title"].fillna("").astype(str) + "\n\n" +
    "Introduction: " + df_tokens.loc[mask_gerado, "Introduction_clean"].fillna("").astype(str) + "\n\n" +
    "Conclusion: " + df_tokens.loc[mask_gerado, "Conclusion_clean"].fillna("").astype(str)
)

# ------------------------------------------------------------
# CONTAGEM DE TOKENS
# ------------------------------------------------------------

df_tokens["input_tokens"] = df_tokens["input_text_for_tokens"].apply(count_tokens)

# ------------------------------------------------------------
# RESUMO GERAL
# ------------------------------------------------------------

summary_tokens = (
    df_tokens[df_tokens["Class"].isin(["Polida_IA", "Gerada"])]
    .groupby("Class")
    .agg(
        registros=("input_tokens", "count"),
        total_tokens=("input_tokens", "sum"),
        media_tokens=("input_tokens", "mean"),
        mediana_tokens=("input_tokens", "median"),
        min_tokens=("input_tokens", "min"),
        max_tokens=("input_tokens", "max")
    )
    .reset_index()
)

summary_tokens

,Class,registros,total_tokens,media_tokens,mediana_tokens,min_tokens,max_tokens
0,Gerada,2357,3162341,1341.680526,1285.0,255,5580
1,Polida_IA,2357,454182,192.694951,181.0,3,584


In [ ]:
# ============================================================
# TESTE UNITÁRIO — 8 AMOSTRAS
# 4 Gerada + 4 Polida_IA
# Modelo: GPT-5.4
# ============================================================

import pandas as pd
import time
from openai import OpenAI
from tqdm import tqdm

client = OpenAI(api_key="")

MODEL = "gpt-5.4"

# ------------------------------------------------------------
# 1. Amostragem
# ------------------------------------------------------------

df_gerada_test = (
    df[df["Class"].eq("Gerada")]
    .sample(n=4, random_state=42)
    .copy()
)

df_polida_test = (
    df[df["Class"].eq("Polida_IA")]
    .sample(n=4, random_state=42)
    .copy()
)

# ------------------------------------------------------------
# 2. Prompt para classe Gerada
# ------------------------------------------------------------

def build_prompt_gerada(row, target_words=180):
    return f"""
Escreva um resumo científico em português do Brasil com base apenas nas informações fornecidas.

Não adicione informações externas.
Não invente métodos, resultados ou conclusões.
Use linguagem acadêmica formal.
O resumo deve ter aproximadamente {target_words} palavras.
Retorne apenas o resumo.

Título:
{row["Title"]}

Introdução:
{row["Introduction"]}

Conclusão:
{row["Conclusion"]}
""".strip()

# ------------------------------------------------------------
# 3. Prompts curtos para classe Polida_IA
# ------------------------------------------------------------

POLISH_PROMPTS = [
    "Melhore a escrita deste resumo acadêmico:",
    "Reescreva este resumo de forma mais clara e acadêmica:",
    "Corrija e melhore a redação deste texto acadêmico:",
    "Deixe este resumo mais fluido e formal:"
]

def build_prompt_polida(text, prompt_instruction):
    return f"""
{prompt_instruction}

{text}
""".strip()

# ------------------------------------------------------------
# 4. Função de chamada à API
# ------------------------------------------------------------

def call_gpt(prompt):
    response = client.responses.create(
        model=MODEL,
        input=prompt,
        temperature=0.4,
        max_output_tokens=500
    )
    return response.output_text.strip()

# ------------------------------------------------------------
# 5. Execução dos testes
# ------------------------------------------------------------

results = []

# Classe Gerada
for _, row in tqdm(df_gerada_test.iterrows(), total=len(df_gerada_test), desc="Gerada"):
    prompt = build_prompt_gerada(row)

    try:
        output = call_gpt(prompt)
        error = None
    except Exception as e:
        output = None
        error = str(e)

    results.append({
        "index": row["index"],
        "Class": row["Class"],
        "prompt_id": "gerada_fixo",
        "prompt_text": prompt,
        "input_title": row["Title"],
        "input_abstract": row["Abstract"],
        "input_introduction": row["Introduction"],
        "input_conclusion": row["Conclusion"],
        "output": output,
        "error": error
    })

    time.sleep(1)

# Classe Polida_IA
for i, (_, row) in enumerate(tqdm(df_polida_test.iterrows(), total=len(df_polida_test), desc="Polida_IA")):
    prompt_instruction = POLISH_PROMPTS[i % len(POLISH_PROMPTS)]
    prompt = build_prompt_polida(row["Abstract"], prompt_instruction)

    try:
        output = call_gpt(prompt)
        error = None
    except Exception as e:
        output = None
        error = str(e)

    results.append({
        "index": row["index"],
        "Class": row["Class"],
        "prompt_id": f"polida_prompt_{i+1}",
        "prompt_instruction": prompt_instruction,
        "prompt_text": prompt,
        "input_title": row["Title"],
        "input_abstract": row["Abstract"],
        "input_introduction": row["Introduction"],
        "input_conclusion": row["Conclusion"],
        "output": output,
        "error": error
    })

    time.sleep(1)

# ------------------------------------------------------------
# 6. Consolidar resultados
# ------------------------------------------------------------

df_test_results = pd.DataFrame(results)

df_test_results["output_words"] = (
    df_test_results["output"]
    .fillna("")
    .str.split()
    .str.len()
)

df_test_results.to_csv(
    "teste_unitario_gpt54_8_amostras.csv",
    index=False,
    encoding="utf-8-sig"
)

df_test_results[[
    "index",
    "Class",
    "prompt_id",
    "output_words",
    "error",
    "output"
]]